# Hyperparameter optimisation

In this notebook we will explore hyperparameter optimisation, and consider a couple of methods for doing so. The goal is for you to know more about what happens under the hood.

## Import and set up

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os

if os.getenv("COLAB_RELEASE_TAG") is None:
    DATA_IN_PATH = '../data'
else:
    DATA_IN_PATH = 'https://raw.githubusercontent.com/rguilcas/BCCR-ML-course/refs/heads/hpo-exercises/lecture_exercises/data'

score_label = 'Validation accuracy'

In [ ]:
def decorate_plot(ax, xlabel=None, ylabel=None, legend=False):
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.spines[['right', 'top']].set_visible(False)
    if legend:
        ax.legend(frameon=False)

def double_plot(ax, xx, yy, col, label, ll=None, aa=1):
    # Do the plotting twice, because we only want to add to the legend once
    ax.plot(xx, yy.T.iloc[0].T, c=col, linestyle=ll, alpha=aa,
                label=label);
    ax.plot(xx, yy, c=col, linestyle=ll, alpha=aa,);

## **Part I: Exploring the results of a hyperparameter search**
### **I.1. Explore the data**
Running a hyperparameter sweep to collect the performance of many hyperparameters can be time-consuming. To save time for this session, we will rely on the results of a previous saved sweep, that was done on the weather image classification project. 
Here, we load the scores of 64 different iterations of the neural network training with different sets of hyperparameters. The results are stored as a **.csv** file.

In [ ]:
# First we load in the validation accuracy for the runs over successive epochs
score_df = pd.read_csv(DATA_IN_PATH+'/hpo-sweeps/pandas-8-score_df.csv', index_col=0).T.reset_index(drop=True)
score_df.index = score_df.index.rename('epoch')
score_df = score_df.T

score_df

Each column of the pandas dataframe corresponds to the validation accuracy of the neural network over different training epochs. Let's plot these validation accuracy curves to see the temporal evolution:

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(10,4), width_ratios=[3,1])
score_df.T.plot(legend=False, ax=axs[0], linewidth=.5)
axs[0].set_title('Validation accuracy over epochs\nColors represent individual runs')
decorate_plot(axs[0], 'Epoch', score_label, legend=False)

axs[1].hist(score_df.T.iloc[-1,], label='All runs', alpha=0.5, bins=np.arange(10, 71, 2.5), density=True, orientation='horizontal', color='.3')
axs[1].set_ylabel('Distribution of Accuracy at last epoch')
axs[1].tick_params(labelleft=False, bottom=False, labelbottom=False)

### **I.2. Best hyperparameter sets**
We will now look at the sets of hyperparameters that lead to the best accuracy results.
To do this, we start by loading the configuration of each run that gives us the hyperparameters that were used. We then explore how they influence the best accuracy obtained for each run.

In [ ]:
df_hyperparams = pd.read_json(DATA_IN_PATH+'/hpo-sweeps/pandas-8-config_dict.json').T
best_accuracy_per_run = score_df.max(axis=1)
df_hyperparams['best_accuracy'] = best_accuracy_per_run

df_hyperparams

The different hyperparameters correspond to the following:

| Hyperparameter name   |      Meaning                                            |
|:---------------------:|:-------------------------------------------------------:|
| C2                    | Number of channels after the first convolutional layer  | 
| C3                    | Number of channels after the second convolutional layer | 
| lr                    | Learning rate                                           |  
| L1_expo               | Number of neurons in the first fully connected layer    |   
| L2_expo               | Number of neurons in the second fully connected layer   |   
| optimizer             | Optimizer used to update weights and biases             | 


> #### **Question 1**
> Which hyperparameters give the 5 best accuracies?
>
> Use the pandas dataframe method `dataframe.nlargest(number_of_elements, sorting_column)`
>
> What is consistent between them? What is different?

In [ ]:
top_5_runs = ...

,C2,C3,lr,L1_expo,L2_expo,optimizer,best_accuracy
0fchbmdb,25,19,0.000045,10,5,Adam,64.8689
ekmw8sv2,20,20,0.000042,6,8,Adam,63.5955
v26pmaf0,32,21,0.000257,4,6,Adam,62.9963
kikwbayq,19,18,0.000164,3,5,AdamW,61.4232
rig4q6ue,26,27,0.000005,10,10,Adam,61.4232


<div class="alert alert-block alert-success">
     We can see that different set of parameters can lead to very similar skill: it is hard to find the optimal set!
</div>

### **I.2. Let's look at the influence of individual hyperparameters**

> #### **Question 2**:
> Fill in the list of hyperparameters we've explored in the list below to inspect their impact on performance. The following cells plots the values of each hyperparameter against the best accuracy obtained. For now, we keep the optimiser separate by changing color based on the optimiser. 

In [ ]:
col_dict = {'SGD': 'C0',
           'Adam': 'C1',
           'AdamW': 'C2'}

for HP in [ ... ]:
    fig, ax = plt.subplots()
    for run_id in df_hyperparams.index:
        ax.scatter(df_hyperparams.loc[run_id, HP], 
                   df_hyperparams.loc[run_id, 'best_accuracy'],
                   color=col_dict[df_hyperparams.loc[run_id, 'optimizer']])
    decorate_plot(ax, HP, score_label)
    if HP == 'lr':
        plt.xscale('log')
        ax.set_xlabel(HP + ' (log axis)')

> #### **Question 3**:
> What conclusions can you draw from the above plots? Which hyperparameters seem the most important?

### **I.3. Let's focus on optimizers**
For the optimizers we can compare histograms of the final performance. The following cell produces a histogram of best accuracy for the three optimizers tested.

> #### **Question 4**:
> Fill in the missing code to create the necessary lists of run_ids and a set of histograms of the performance of each optimiser.

In [ ]:
# Separate the runs by which optimiser was used
sgd_runs = df_hyperparams.loc[df_hyperparams.optimizer == 'SGD'].index
adam_runs = ...
adamw_runs = ...

In [ ]:
bins = np.linspace(15, 65, 21)
fig, axes = plt.subplots(3, 1)
ii = 0
for label, sub_list in [('SGD', sgd_runs), ('Adam', adam_runs), ('AdamW', adamw_runs)]:
    ax = axes[ii]
    ax.hist(df_hyperparams.loc[sub_list, 'best_accuracy'], label=label, alpha=0.5, bins=bins,
             density=True, color=col_dict[label])
    decorate_plot(ax, score_label, None, legend=True)
    ax.set_ylim([0, 0.15])
    ii += 1

Because we picked the optimiser randomly for each run we do not have an equal amount of runs for each optimiser. 

In [ ]:
for label, sub_list in [('SGD', sgd_runs), ('Adam', adam_runs), ('AdamW', adamw_runs)]:
    print(label, len(sub_list))

> #### **Question 5**:
> If you were to set up more experiments which optimisers would you use? Why?

## **Part II: Example Hyperparameter Optimisation approaches**
### **II.1. Successive halving**

The successive halving algorithm is designed to greedily discard low-performing models, so that the hyperparamter optimisation budget can be focussed on high-performing configurations. To do so, at every decision step it only keeps the best 1/**H** runs, and discards the rest. It will then run the remaining runs for **H** times as long as for the previous choice. Here we will use **H**=2, so we will discard the bottom half, and for each such discarding choice double the number of epochs trained before the next decision point. If you want to learn more, see e.g. https://blog.ml.cmu.edu/2018/12/12/massively-parallel-hyperparameter-optimization/

As before, we will take advantage of the existing run that we have on record in order to speed up our exploration, and just pretend we get the results epoch per epoch.

> #### **Question 6:**
> Fill in the missing parts of the code so we can implement successive halving successfully. You have to complete the `[...TODO...]` written parts.

In [ ]:
discarder_runs_list, median_accuracy_list, epochs_where_splitting_list = [], [], [] # We want to store some intermediate states
df_runs_kept_previous_step = score_df
current_epoch = 0 # Tracking which epoch we're using.
main_fig, main_ax = plt.subplots()
for spliting_step in range(1, 5):
    current_epoch = current_epoch + 2**(spliting_step-1) # Update which epoch we'll look at when splitting
    epochs_where_splitting_list.append(current_epoch) # Record keeping
    current_step_median_accuracy = df_runs_kept_previous_step[current_epoch].median() # Take the middle value as the threshold
    median_accuracy_list.append(current_step_median_accuracy)
    
    # Create two new data frames, one for the runs we're keeping, and one for the one we're dropping
    df_runs_kept_previous_current = df_runs_kept_previous_step[ ...TODO... ] # FILL IN THE ANSWER HERE
    df_runs_discarded_current = df_runs_kept_previous_step[ ...TODO... ] # FILL IN THE ANSWER HERE
    discarder_runs_list.append(df_runs_discarded_current) # Track all the runs we're discarding

    df_runs_kept_previous_step = df_runs_kept_previous_current # Update variable
    print(f'Epoch {current_epoch}, threshold: {current_step_median_accuracy}, num runs left: {len(df_runs_kept_previous_current)}')

    # Add the discarded runs to our main figure
    # Do the plotting twice, because we only want to add to the legend once
    col = f'C{spliting_step-1}'
    double_plot(main_ax, range(current_epoch+1), df_runs_discarded_current.T[:current_epoch+1], col=col,
                label=f'Discarded epoch {current_epoch}')

    # Also make a figure for this decision point
    fig, ax = plt.subplots()
    double_plot(ax, range(current_epoch+1), df_runs_discarded_current.T[:current_epoch+1], col=col, label='Discarded: observed')
    double_plot(ax, range(21), df_runs_discarded_current.T, col=col, label='Discarded: future',
                ll=':', aa=0.5)
    double_plot(ax, range(current_epoch+1), df_runs_kept_previous_current.T[:current_epoch+1], label='Kept: observed', col='k', aa=0.5)
    double_plot(ax, range(21), df_runs_kept_previous_current.T, col='k', label='Kept: future', ll=':', aa=0.5)
    decorate_plot(ax, 'Epoch', score_label, legend=True)
    ax.set_title(f'Step {spliting_step}: splitting at epoch {current_epoch} with threshold {current_step_median_accuracy:.2f}')

kept_final = df_runs_kept_previous_current
double_plot(main_ax, range(21), kept_final.T, col='k', label='Kept')
decorate_plot(main_ax, 'Epoch', score_label, legend=True)


> #### **Question 7**: 
> Do you think the algoritm is reasonable in the runs that it stops early?

### **II.1. Learning curves**
An alternative approach is to treat the evolution of the training curves as a prediction problem, and train a simpler machine learning model to predict the evolution over epochs of the validation accuracy. Instead of discarding the runs that are have the lowest accuracy at a given step, we discard the runs with the lowest predicted accuracy in the future.

Here we will train linear regression models to predict future validation accuracies based on the first **K** epochs. These could then be used to decide which runs to continue and which to stop early.

In [ ]:
from sklearn.linear_model import LinearRegression

> #### **Question 8**:
> Fill in the remaining code so we can predict future accuracy.

In [ ]:
K = 5 # How many epochs we run for before predicting the evolution. Used to split into X and y.
X = score_df.T[:K].T
y = score_df.T[K:].T

N = 40 # How many runs to use for training. Used to split into training and validation subsets.
X_train = X[:N]
y_train = y[:N]
X_val = X[N:]
y_val = y[N:]

# We fit a linear regression model to the training data
reg = LinearRegression().fit(..., ...)

In [ ]:
# Plot the predicted validation curves
fig, ax = plt.subplots()
for ii in range(10):
    X_ii = X_val.iloc[[ii]]
    y_ii = y_val.iloc[[ii]]
    ax.plot(range(K), X_ii.values[0], '--', c=f'C{ii}',
            label='Features' if ii == 0 else None);
    ax.plot(range(K, 21), reg.predict(X_ii).T, ':', c=f'C{ii}',
            label='Predicted' if ii == 0 else None);
    ax.plot(range(K, 21), y_ii.values[0], c=f'C{ii}',
            label='True' if ii == 0 else None);
ax.set_xlabel('Epoch')
ax.set_ylabel(score_label)
ax.spines[['right', 'top']].set_visible(False)
ax.set_title('Linear regression to predict future accuracy')
ax.legend(frameon=False);

> #### **Question 9**:
> How well do the predictions work? What are they good at? How much computation could you save by using the predictions as heuristics for final performance to limit the number of models trained to end? 
>
>What happens if you change **K** and **N**?